A notebook for batch processing the ExoMars SKP Field Test 1 dataset into RGB products, using the fully automated '99u' (99th percentile unbalanced) stretch routine.

In [2]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'svg' # note - requires Inkscape to be installed, and to be accessible from the terminal
# on macos, add inkscape to the path with `sudo ln -s /Applications/Inkscape.app/Contents/MacOS/inkscape /usr/local/bin/inkscape`
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import aupy

# Setting Paths and Arranging the Data

Data from the ROCC comes in various directory structures, and with the filenames relabelled.

We need to reorganise and relabel this data, which can all be done via accessing the xml files attached with each image.

In steps:

1. Access directory for all data from the given sol.
2. Find all image data (as png's) within the sol directory.
3. Find the paired xml files for each image.
4. Extract filenames and other organisational metadata from the images and xml files.
5. Put the data into a new processing directory for that sol.

In [4]:
from pathlib import Path

In [5]:
data_dir = Path('/Users/sci/Documents/Documents - Roger’s MacBook Air/projects/esaskp_2025-2027/field_trials/SKP_FT1/data/downlink')

In [6]:
# make a list of all the directories in the data directory
sols = [d.name for d in data_dir.iterdir()]
# drop .DS_Store
sols = [s for s in sols if s != '.DS_Store']
sols.sort()

In [7]:
sols

['sol03',
 'sol04',
 'sol05',
 'sol06',
 'sol07',
 'sol38',
 'sol39',
 'sol40',
 'sol41']

Choose a Sol to process.

Let's develop on Sol41, because we have a range of cameras and scenes on this sol.

In [8]:
sol = sols[-1]
sol

'sol41'

In [13]:
# check that the sol directory exists
sol_dir = data_dir / sol
if not sol_dir.exists():
    raise FileNotFoundError(f"Sol directory {sol_dir} does not exist.")

The downlink will be arranged in various directories that might not find a pattern.

A clean way to deal with this is bypass the folders, and just search for the png.s

In [21]:
def get_downlinked_dirs(sol_dir):
    dl_dirs = [d.name for d in sol_dir.iterdir() if d.is_dir()]
    return sorted(dl_dirs)

In [22]:
# get the downlinked directories in the sol directory
dl_dirs = get_downlinked_dirs(sol_dir)
dl_dirs

['20260924T081817UTC_sol41_part1',
 '20260924T095233UTC_sol41_part2',
 'sol41_clupi',
 'sol41_enfys_only',
 'sol41_hrc_png']

In [28]:
import re
from pathlib import Path
from typing import List

def get_all_pngs(sol_dir: str, dl_dirs: List[str]):
    """use pathlib and regex to find all png files in the 
    downlinked directories recursively"""

    png_files = []
    for dl_dir in dl_dirs:
        dl_path = Path(sol_dir) / dl_dir
        for file_path in dl_path.rglob("*.png"):
            if re.match(r".*\.png$", file_path.name):
                png_files.append(file_path)
    png_files.sort()
    return png_files

In [29]:
png_files = get_all_pngs(sol_dir, dl_dirs)

In [30]:
png_files

[PosixPath('/Users/sci/Documents/Documents - Roger’s MacBook Air/projects/esaskp_2025-2027/field_trials/SKP_FT1/data/downlink/sol41/20260924T081817UTC_sol41_part1/LOCCAM_LEFT_1790239410114.png'),
 PosixPath('/Users/sci/Documents/Documents - Roger’s MacBook Air/projects/esaskp_2025-2027/field_trials/SKP_FT1/data/downlink/sol41/20260924T081817UTC_sol41_part1/LOCCAM_RIGHT_1790239410261.png'),
 PosixPath('/Users/sci/Documents/Documents - Roger’s MacBook Air/projects/esaskp_2025-2027/field_trials/SKP_FT1/data/downlink/sol41/20260924T081817UTC_sol41_part1/NAVCAM_LEFT_1790239345356.png'),
 PosixPath('/Users/sci/Documents/Documents - Roger’s MacBook Air/projects/esaskp_2025-2027/field_trials/SKP_FT1/data/downlink/sol41/20260924T081817UTC_sol41_part1/NAVCAM_LEFT_1790239370200.png'),
 PosixPath('/Users/sci/Documents/Documents - Roger’s MacBook Air/projects/esaskp_2025-2027/field_trials/SKP_FT1/data/downlink/sol41/20260924T081817UTC_sol41_part1/NAVCAM_LEFT_1790239390354.png'),
 PosixPath('/Users/

In [42]:
def get_png_xmls(png_files: List[Path]):
    """Pair each png file with an xml file.
    If no xml is present, file with None"""
    png_xml_pairs = []  # List to store tuples of (png_file, xml_file or None)
    for png_file in png_files:
        xml_file = Path(str(png_file) + ".xml")
        if xml_file.exists():
            png_xml_pairs.append((png_file, xml_file))
        else:
            print(f"Warning: XML file not found for {png_file.stem}. Dropping this file from processing")            

    return png_xml_pairs

In [43]:
png_xml_pairs = get_png_xmls(png_files)

Loop over the list of png-xml pairs, and for each, access the xml file.

In [48]:
xml_file

PosixPath('/Users/sci/Documents/Documents - Roger’s MacBook Air/projects/esaskp_2025-2027/field_trials/SKP_FT1/data/downlink/sol41/20260924T081817UTC_sol41_part1/LOCCAM_LEFT_1790239410114.png.xml')

In [49]:
meta_dict = aupy.load_xml_metadata(xml_file)
pd.read_xml(xml_path, xpath=".//attribute", parser = "etree")

KeyboardInterrupt: 

In [46]:
for png_file, xml_file in png_xml_pairs:
    # read the xml file with pandas
    meta_dict = aupy.load_xml_metadata(xml_file)

/opt/homebrew/Caskroom/miniconda/base/envs/aupy_dev/lib/python3.13/site-packages/IPython/extensions/deduperreload/deduperreload.py:290: DeprecationWarning: ast.Ellipsis is deprecated and will be removed in Python 3.14; use ast.Constant instead
  elif not isinstance(ast_elt, (ast.Ellipsis, ast.Pass)):
/opt/homebrew/Caskroom/miniconda/base/envs/aupy_dev/lib/python3.13/site-packages/IPython/extensions/deduperreload/deduperreload.py:290: DeprecationWarning: ast.Ellipsis is deprecated and will be removed in Python 3.14; use ast.Constant instead
  elif not isinstance(ast_elt, (ast.Ellipsis, ast.Pass)):
/opt/homebrew/Caskroom/miniconda/base/envs/aupy_dev/lib/python3.13/site-packages/IPython/extensions/deduperreload/deduperreload.py:290: DeprecationWarning: ast.Ellipsis is deprecated and will be removed in Python 3.14; use ast.Constant instead
  elif not isinstance(ast_elt, (ast.Ellipsis, ast.Pass)):
/opt/homebrew/Caskroom/miniconda/base/envs/aupy_dev/lib/python3.13/site-packages/IPython/exten

KeyboardInterrupt: 

Choose a scene to process.

In [36]:
scene = scenes[-2]
scene

'sol39_postdrive_forward_wac_rrgb'

Check if the scene contains any trial directories

In [37]:
# look for subdirectorys in the scene directory
scene_dir = sol_dir / scene
# check that the scene directory exists
if not scene_dir.exists():
    raise FileNotFoundError(f"Scene directory {scene_dir} does not exist.")
# get the subdirectories in the scene directory
trials = [d.name for d in scene_dir.iterdir() if d.is_dir()]
trials.sort()

if trials == []:
    trials = ['']

trials

['caltarg',
 'pos01',
 'pos02',
 'pos03',
 'pos04',
 'pos05',
 'pos06',
 'pos07',
 'pos08',
 'pos09',
 'pos10',
 'pos11',
 'pos12']

The batch script

In [47]:
# for sol in sols:
#     # get the scenes in the sol directory
#     sol_dir = data_dir / sol
#     # check that the sol directory exists
#     if not sol_dir.exists():
#         raise FileNotFoundError(f"Sol directory {sol_dir} does not exist.")
#     # get the scenes in the sol directory
#     scenes = [d.name for d in sol_dir.iterdir() if d.is_dir()]
#     scenes.sort()
    
#     print(f'{sol}: {scenes}')
#     for scene in scenes:

# look for subdirectorys in the scene directory
scene_dir = sol_dir / scene
# check that the scene directory exists
if not scene_dir.exists():
    raise FileNotFoundError(f"Scene directory {scene_dir} does not exist.")
# get the subdirectories in the scene directory
trials = [d.name for d in scene_dir.iterdir() if d.is_dir()]
trials.sort()

if trials == []:
    trials = [None]

print(f'  {sol} {scene}: {trials}')        
for trial in trials:
    print(f'    {sol} {scene} {trial}')

    # # process HRC
    # hrc_loader = aupy.AupeIO('HRC', 'RGB',scene_dir, sol, scene, trial=trial)
    # if hrc_loader.input_files != []:            
    #     hrc_rgb = hrc_loader.load_frame()
    #     hrc_rgb.debayer()
    #     hrc_rgb.export_image('99s')
    #     hrc_rgb.export_image('bps')

    # process LWAC
    lwac_loader = aupy.AupeIO('LWAC', 'RGB',scene_dir, sol, scene, trial=trial)
    if lwac_loader.input_files != []:
        lwac_rgb = lwac_loader.load_frame()
        lwac_rgb.export_image('bpu', gamma=2.2)
        # lwac_rgb.export_image('bpb')

    # # process RWAC
    # rwac_loader = aupy.AupeIO('RWAC', 'RGB',scene_dir, sol, scene, trial=trial)
    # if rwac_loader.input_files != []:
    #     rwac_rgb = rwac_loader.load_frame()            
    #     rwac_rgb.export_image('99u')
    #     rwac_rgb.export_image('bpb')


  sol39 sol39_postdrive_forward_wac_rrgb: ['caltarg', 'pos01', 'pos02', 'pos03', 'pos04', 'pos05', 'pos06', 'pos07', 'pos08', 'pos09', 'pos10', 'pos11', 'pos12']
    sol39 sol39_postdrive_forward_wac_rrgb caltarg
Exporting image using brightest pixel unbalanced
Stretching image using brightest pixel unbalanced
    sol39 sol39_postdrive_forward_wac_rrgb pos01
Exporting image using brightest pixel unbalanced
Stretching image using brightest pixel unbalanced
    sol39 sol39_postdrive_forward_wac_rrgb pos02
Exporting image using brightest pixel unbalanced
Stretching image using brightest pixel unbalanced
    sol39 sol39_postdrive_forward_wac_rrgb pos03
Exporting image using brightest pixel unbalanced
Stretching image using brightest pixel unbalanced
    sol39 sol39_postdrive_forward_wac_rrgb pos04
Exporting image using brightest pixel unbalanced
Stretching image using brightest pixel unbalanced
    sol39 sol39_postdrive_forward_wac_rrgb pos05
Exporting image using brightest pixel unbalance